In [2]:
import pandas as pd
import numpy as np
import os
import torch
import warnings
import random
import joblib

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from sklearn.metrics import pairwise_distances_argmin
from scipy.stats import ttest_ind
from sklearn.covariance import LedoitWolf

import matplotlib.pyplot as plt
import umap
import nltk
import torch
import torch.nn.functional as F
import hdbscan
from transformers import AutoTokenizer

#nltk.download('punkt')
#nltk.download('punkt_tab')


"""   

Copyright (c) 2026, Michael Tchuindjang
All rights reserved.

This code was developed as part of a PhD research project in Cybersecurity and Artificial Intelligence, 
supported by a studentship at the University of the West of England (UWE Bristol).

Use of this software is permitted for academic, educational, and research purposes.  
For any commercial use or redistribution, please contact the author for permission.

Disclaimer:
In no event shall the author or UWE be liable for any claim, damages, or other liability arising from the use of this code.

Acknowledgment of the author and the research context is appreciated in any derivative work or publication.


"""



# =========================
# GLOBAL PARAMETERS
# =========================
TRAIN_DIR = "training"
os.makedirs(TRAIN_DIR, exist_ok=True)

# We automatically create the folder for ablation if missed
FOLDER = "cosine_mahalanobis"
os.makedirs(FOLDER, exist_ok=True)

TESTS = ['Test_1', 
         'Test_2', 
         'Test_3', 
         'Test_4']

TRAINING_TEST = TESTS[0] # TESTS[1]: Test_2
# Tests/training files are formatted like Test_X_all_models.csv
TRAINING_TEST_FILE = TRAINING_TEST + '_all_models.csv'
INPUT_FILE = os.path.join(TRAIN_DIR, TRAINING_TEST_FILE)

EMB_MODEL_NAMES = [
    'all-MiniLM-L6-v2',
    'all-mpnet-base-v2',
    'all-roberta-large-v1'
]

EMBEDDING_MODEL_NAME = EMB_MODEL_NAMES[0]

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/" + EMBEDDING_MODEL_NAME)

CHUNKING_POOLING = "norm_mean" # change: mean / weighted / max / norm_mean
CHUNKING_OVERLAP = 40  # 0, 20, 40, 60

USE_CHUNKING = False

CHUNK_TAG = "chunked" if USE_CHUNKING else "no_chunk"

USE_FULL_CONVERSATION = False

RUN_SIGNATURE = f"{EMBEDDING_MODEL_NAME}__{CHUNK_TAG}__full-{USE_FULL_CONVERSATION}__{TRAINING_TEST}"


# =========================
# TEXT EXTRACTION
# =========================
def get_final_response(row):
    return row.get(f"output_turn_{row['turn_depth']}", "")

def get_full_conversation(row):
    conversation = []
    for i in range(1, row["turn_depth"] + 1):
        user_text = row.get(f"turn_{i}", "")
        assistant_text = row.get(f"output_turn_{i}", "")

        if pd.notna(user_text) and user_text.strip():
            conversation.append(f"USER: {user_text}")

        if pd.notna(assistant_text) and assistant_text.strip():
            conversation.append(f"ASSISTANT: {assistant_text}")

    return "\n".join(conversation)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # makes some ops deterministic (optional but good for research)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =========================================================
# 1. LOAD MODELS
# =========================================================
models = {
    "MiniLM": {
        "model": SentenceTransformer("all-MiniLM-L6-v2"),
        "name": "all-MiniLM-L6-v2"
    },
    "MPNet": {
        "model": SentenceTransformer("all-mpnet-base-v2"),
        "name": "all-mpnet-base-v2"
    },
    "RoBERTa": {
        "model": SentenceTransformer("all-roberta-large-v1"),
        "name": "all-roberta-large-v1"
    }
}


# =========================
# CHUNKING ENCODER
# =========================
def embed_no_chunk(texts, model, tokenizer):
    """
    Encode list of texts using standard truncated embedding.
    
    Args:
        texts (list[str])
        model (SentenceTransformer)

    Returns:
        np.ndarray: shape (N, D)
    """
    return model.encode(texts, convert_to_numpy=True, batch_size=64, show_progress_bar=True)


def chunk_text_token_level(text, tokenizer, max_tokens=256, overlap=CHUNKING_OVERLAP):
    """
    Token-consistent chunking using model tokenizer.
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk.strip())

        start += max_tokens - overlap

    return chunks

# =========================================================
# 3. POOLING STRATEGIES (UNBIASED OPTIONS)
# =========================================================

def mean_pool(embeddings):
    return np.mean(embeddings, axis=0)

def weighted_mean_pool(embeddings, chunks, tokenizer):
    weights = np.array([
        len(tokenizer.encode(c, add_special_tokens=False))
        for c in chunks
    ])
    return np.average(embeddings, axis=0, weights=weights)

def max_pool(embeddings):
    return np.max(embeddings, axis=0)

def normalized_mean_pool(embeddings):
    emb = normalize(embeddings, axis=1)
    pooled = np.mean(emb, axis=0)
    return normalize(pooled.reshape(1, -1))[0]

# POOLING_METHOD is "weighted" by default  # change: mean / weighted / max / norm_mean
def embed_chunk(texts, model, tokenizer, pooling=CHUNKING_POOLING, batch_size=64):
    all_embeddings = []
    chunk_counts = []
    all_chunks = []

    # -----------------------------
    # Flatten chunks
    # -----------------------------
    for text in texts:
        chunks = chunk_text_token_level(text, tokenizer)

        if len(chunks) == 0:
            chunks = [""]

        chunk_counts.append(len(chunks))
        all_chunks.extend(chunks)

    # -----------------------------
    # Encode chunks
    # -----------------------------
    chunk_embeddings = model.encode(
        all_chunks,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    # -----------------------------
    # Pool per document
    # -----------------------------
    idx = 0

    for i, count in enumerate(chunk_counts):
        doc_chunks = chunk_embeddings[idx:idx + count]
        idx += count

        if pooling == "mean":
            doc_emb = mean_pool(doc_chunks)

        elif pooling == "weighted":
            doc_emb = weighted_mean_pool(doc_chunks, all_chunks[:count], tokenizer)

        elif pooling == "max":
            doc_emb = max_pool(doc_chunks)

        elif pooling == "norm_mean":
            doc_emb = normalized_mean_pool(doc_chunks)

        else:
            raise ValueError("Unknown pooling method")

        all_embeddings.append(doc_emb)

    return np.array(all_embeddings)


# =========================
# ENCODING WRAPPER
# =========================
def encode_texts(texts, model, tokenizer):
    if USE_CHUNKING:
        embeddings = embed_chunk(texts, model, tokenizer)
        if isinstance(embeddings, torch.Tensor):
            embeddings = embeddings.detach().cpu().numpy()
        return embeddings
    return embed_no_chunk(texts, model, tokenizer)


# =========================
# COSINE DISTRIBUTIONS
# =========================
def cosine_distributions(embeddings, labels):
    sim_matrix = cosine_similarity(embeddings)

    safe_safe, harm_harm, safe_harm = [], [], []

    N = len(labels)

    for i in range(N):
        for j in range(i + 1, N):
            if labels[i] == labels[j] == "safe":
                safe_safe.append(sim_matrix[i, j])
            elif labels[i] == labels[j] == "harmful":
                harm_harm.append(sim_matrix[i, j])
            else:
                safe_harm.append(sim_matrix[i, j])

    return safe_safe, harm_harm, safe_harm


# =========================
# MAHALANOBIS DISTANCE
# =========================
def mahalanobis_distances(embeddings, labels):
    """
    Compute distance of each point to global distribution.
    Lower = more typical
    Higher = more anomalous
    """

    cov_estimator = LedoitWolf().fit(embeddings)
    precision = cov_estimator.precision_

    mean = np.mean(embeddings, axis=0)

    def maha(x):
        diff = x - mean
        return np.sqrt(diff @ precision @ diff.T)

    distances = np.array([maha(x) for x in embeddings])

    safe_dist = distances[np.array(labels) == "safe"]
    harm_dist = distances[np.array(labels) == "harmful"]

    return safe_dist, harm_dist


# =========================
# STATISTICS
# =========================
def compute_stats(safe_safe, harm_harm, safe_harm, safe_maha, harm_maha):
    intra_cos = safe_safe + harm_harm
    inter_cos = safe_harm

    stats = {
        # cosine
        "cos_intra_mean": np.mean(intra_cos),
        "cos_inter_mean": np.mean(inter_cos),

        # mahalanobis
        "maha_safe_mean": np.mean(safe_maha),
        "maha_harm_mean": np.mean(harm_maha),
    }

    # significance tests
    cos_t, cos_p = ttest_ind(intra_cos, inter_cos, equal_var=False)
    maha_t, maha_p = ttest_ind(safe_maha, harm_maha, equal_var=False)

    stats.update({
        "cos_t_stat": cos_t,
        "cos_p_value": cos_p,
        "maha_t_stat": maha_t,
        "maha_p_value": maha_p,
    })

    return stats


# =========================
# 4. PLOTTING
# =========================
def plot_comparison(safe_safe, harm_harm, safe_harm, safe_maha, harm_maha, output_path):
    plt.figure(figsize=(10, 4))

    # Cosine subplot
    plt.subplot(1, 2, 1)
    plt.hist(safe_safe, bins=40, alpha=0.5, label="safe-safe")
    plt.hist(harm_harm, bins=40, alpha=0.5, label="harm-harm")
    plt.hist(safe_harm, bins=40, alpha=0.5, label="safe-harm")
    plt.title("Cosine Similarity")
    plt.legend()

    # Mahalanobis subplot
    plt.subplot(1, 2, 2)
    plt.hist(safe_maha, bins=40, alpha=0.5, label="safe")
    plt.hist(harm_maha, bins=40, alpha=0.5, label="harmful")
    plt.title("Mahalanobis Distance")
    plt.legend()

    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


def save_stats_csv(stats, output_path):
    
    # Normalize to DataFrame
    if isinstance(stats, pd.DataFrame):
        df = stats

    elif isinstance(stats, dict):
        df = pd.DataFrame([stats])

    elif isinstance(stats, list):
        # list of dicts OR list of lists
        df = pd.DataFrame(stats)

    else:
        raise ValueError(f"Unsupported stats type: {type(stats)}")

    file_exists = os.path.isfile(output_path)

    df.to_csv(
        output_path,
        mode="a",
        header=not file_exists,
        index=False
    )




def plot_umap_embeddings(embeddings, labels, title, save_path):
    """
    UMAP projection of embeddings with safe vs harmful labels
    """

    X = np.array(embeddings)
    y = np.array([0 if l == "safe" else 1 for l in labels])

    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=42
    )

    X_2d = reducer.fit_transform(X)

    plt.figure(figsize=(8, 6))

    plt.scatter(
        X_2d[y == 0, 0],
        X_2d[y == 0, 1],
        label="Safe",
        alpha=0.6
    )

    plt.scatter(
        X_2d[y == 1, 0],
        X_2d[y == 1, 1],
        label="Harmful",
        alpha=0.6
    )

    plt.title(title)
    plt.legend()
    plt.tight_layout()

    plt.savefig(save_path, dpi=300)
    plt.close()

# =========================
# FULL PIPELINE
# =========================
def run_analysis(
    safe_texts,
    harmful_texts,
):

    texts = safe_texts + harmful_texts
    labels = ["safe"] * len(safe_texts) + ["harmful"] * len(harmful_texts)

    print(f"\n🚀 Running analysis for model: {EMBEDDING_MODEL_NAME}")
    print(f"🧩 Chunking enabled: {USE_CHUNKING}")

    # -------------------------
    # Reproducibility
    # -------------------------
    set_seed(42)

    # -------------------------
    # Embedding
    # -------------------------
    print("🔄 Encoding embeddings...")
        
    embeddings = encode_texts(texts, embedder, tokenizer)
    embeddings = normalize(embeddings, axis=1)

    # -------------------------
    # Cosine analysis
    # -------------------------
    safe_safe, harm_harm, safe_harm = cosine_distributions(embeddings, labels)

    # -------------------------
    # Mahalanobis analysis
    # -------------------------
    safe_maha, harm_maha = mahalanobis_distances(embeddings, labels)

    # -------------------------
    # Stats
    # -------------------------
    stats = compute_stats(
        safe_safe,
        harm_harm,
        safe_harm,
        safe_maha,
        harm_maha
    )

    # add metadata for ablation table
    stats.update({
        "test": TRAINING_TEST,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "chunking": USE_CHUNKING,
        "full_conversation": USE_FULL_CONVERSATION
    })

    plot_path = os.path.join(FOLDER, f"{RUN_SIGNATURE}__cosine_vs_mahalanobis.png")
   

    # -------------------------
    # Save outputs
    # -------------------------
    plot_comparison(
        safe_safe,
        harm_harm,
        safe_harm,
        safe_maha,
        harm_maha,
        plot_path
    )

    # -------------------------
    # Logging
    # -------------------------
    print("\n===== RESULTS =====")
    for k, v in stats.items():
        if isinstance(v, (int, float)):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

    print(f"\n📊 Plot saved to: {plot_path}")
    
    #csv_path = f"{FOLDER}/cosine_mahalanobis_stats.csv"
    #save_stats_csv(stats, f"{FOLDER}/cosine_mahalanobis_stats.csv")
    #print(f"\n✅ Stats saved to: {csv_path}")


    # =========================
    # NEW: UMAP VISUALIZATION
    # =========================
    print("📉 Running UMAP projection...")
    
    umap_path = os.path.join(
        FOLDER,
        f"{RUN_SIGNATURE}__umap_safe_vs_harmful.png"
    )
    
    plot_umap_embeddings(
        embeddings=embeddings,
        labels=labels,
        title=f"UMAP: Safe vs Harmful | {EMBEDDING_MODEL_NAME} ({'chunked' if USE_CHUNKING else 'no_chunk'})",
        save_path=umap_path
    )
    
    print(f"📌 UMAP saved to: {umap_path}")

    return stats


def run_ablation(safe_texts, harmful_texts, models=models):
    """
    Runs full ablation:
    - multiple embedding models
    - with and without chunking
    """

    global EMBEDDING_MODEL_NAME, USE_CHUNKING, CHUNK_TAG
    global EMB_MODEL_FOLDER, RUN_SIGNATURE
    global embedder, tokenizer

    all_results = []

    for model_key, model_info in models.items():

        embedder = model_info["model"]
        model_name = model_info["name"]

        tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/'+model_name)

        for use_chunking in [False, True]:

            print("\n" + "=" * 90)
            print(f"🔬 MODEL: {model_key} | CHUNKING: {use_chunking}")
            print("=" * 90)

            EMBEDDING_MODEL_NAME = model_name
            
            USE_CHUNKING = use_chunking

            CHUNK_TAG = "chunked" if use_chunking else "no_chunk"

            RUN_SIGNATURE = f"{EMBEDDING_MODEL_NAME}__{CHUNK_TAG}__full-{USE_FULL_CONVERSATION}__{TRAINING_TEST}"
            
            stats = run_analysis(
                safe_texts=safe_texts,
                harmful_texts=harmful_texts,
            )

            all_results.append(stats)

    # =========================
    # Save full ablation table
    # =========================
    summary_path = os.path.join(FOLDER, "cosine_mahalanobis_stats.csv")
    save_stats_csv(all_results, summary_path)
    summary_df = pd.DataFrame(all_results)

    print("\n✅ FULL ABLATION COMPLETED")
    print(f"Saved to: {summary_path}")

    return summary_df

# =========================
# RUN
# =========================
print(f"Loading data from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)

# Filter
df_safe = df[df["human_ensemble_judge"] == 0].copy()
df_harmful = df[df["human_ensemble_judge"] == 1].copy()

# Extract text
if USE_FULL_CONVERSATION:
    df_safe["text"] = df_safe.apply(get_full_conversation, axis=1)
    df_harmful["text"] = df_harmful.apply(get_full_conversation, axis=1)
else:
    df_safe["text"] = df_safe.apply(get_final_response, axis=1)
    df_harmful["text"] = df_harmful.apply(get_final_response, axis=1)
    
# Clean
safe_texts = df_safe["text"].dropna().tolist()
harmful_texts = df_harmful["text"].dropna().tolist()

# Run abalation
#run_analysis(safe_texts, harmful_texts)
run_ablation(safe_texts, harmful_texts)

Loading data from training/Test_1_all_models.csv...

🔬 MODEL: MiniLM | CHUNKING: False

🚀 Running analysis for model: all-MiniLM-L6-v2
🧩 Chunking enabled: False
🔄 Encoding embeddings...


Batches:   0%|          | 0/56 [00:00<?, ?it/s]


===== RESULTS =====
cos_intra_mean: 0.23743189871311188
cos_inter_mean: 0.17161035537719727
maha_safe_mean: 16.347959518432617
maha_harm_mean: 19.40715789794922
cos_t_stat: 426.8733
cos_p_value: 0.0000
maha_t_stat: -15.4279
maha_p_value: 0.0000
test: Test_1
embedding_model: all-MiniLM-L6-v2
chunking: 0.0000
full_conversation: 0.0000

📊 Plot saved to: cosine_mahalanobis/all-MiniLM-L6-v2__no_chunk__full-False__Test_1__cosine_vs_mahalanobis.png
📉 Running UMAP projection...


/home/michael/venvs/torch-gpu/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (545 > 512). Running this sequence through the model will result in indexing errors


📌 UMAP saved to: cosine_mahalanobis/all-MiniLM-L6-v2__no_chunk__full-False__Test_1__umap_safe_vs_harmful.png

🔬 MODEL: MiniLM | CHUNKING: True

🚀 Running analysis for model: all-MiniLM-L6-v2
🧩 Chunking enabled: True
🔄 Encoding embeddings...


...

Batches:   0%|          | 0/223 [00:00<?, ?it/s]


===== RESULTS =====
cos_intra_mean: 0.25718024373054504
cos_inter_mean: 0.1817842423915863
maha_safe_mean: 21.584684371948242
maha_harm_mean: 25.533143997192383
cos_t_stat: 543.8431
cos_p_value: 0.0000
maha_t_stat: -13.9960
maha_p_value: 0.0000
test: Test_1
embedding_model: all-roberta-large-v1
chunking: 1.0000
full_conversation: 0.0000

📊 Plot saved to: cosine_mahalanobis/all-roberta-large-v1__chunked__full-False__Test_1__cosine_vs_mahalanobis.png
📉 Running UMAP projection...


/home/michael/venvs/torch-gpu/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


📌 UMAP saved to: cosine_mahalanobis/all-roberta-large-v1__chunked__full-False__Test_1__umap_safe_vs_harmful.png

✅ FULL ABLATION COMPLETED
Saved to: cosine_mahalanobis/cosine_mahalanobis_stats.csv


,cos_intra_mean,cos_inter_mean,maha_safe_mean,maha_harm_mean,cos_t_stat,cos_p_value,maha_t_stat,maha_p_value,test,embedding_model,chunking,full_conversation
0,0.237432,0.171610,16.347960,19.407158,426.873276,0.0,-15.427922,1.218746e-51,Test_1,all-MiniLM-L6-v2,False,False
1,0.268934,0.201221,16.410389,19.130077,429.417195,0.0,-13.022050,9.337338e-38,Test_1,all-MiniLM-L6-v2,True,False
2,0.229373,0.156577,21.197502,25.141277,483.529574,0.0,-14.166283,4.137600e-44,Test_1,all-mpnet-base-v2,False,False
3,0.259070,0.189971,21.135954,24.897547,465.766113,0.0,-13.489009,2.840407e-40,Test_1,all-mpnet-base-v2,True,False
4,0.198679,0.120374,22.098980,27.041481,558.645359,0.0,-17.473725,6.290636e-65,Test_1,all-roberta-large-v1,False,False
5,0.257180,0.181784,21.584684,25.533144,543.843076,0.0,-13.995951,4.047156e-43,Test_1,all-roberta-large-v1,True,False
